# V28 — 3D rocket landing

**How to use:** Runtime > Change runtime type > **T4 GPU** > Save. Then Runtime > **Run all**. That's it.

**Resume:** the training checkpoint lives at `MyDrive/rocket_v28/models/master_v28_latest.zip`. Every time you Run all, Cell 3 picks up where it left off — no code edits, no pasting commands. If the file is not there yet (first run, or a different Google account), training starts from step 0.

**Different Google account:** mount Drive here with the account whose `MyDrive/rocket_v28/` holds the checkpoint. If you want to keep running from another account, share that folder with the new account in Drive (right-click > Share) before you Run all the first time on the new account.

## Cell 1: setup — GPU, Drive, dependencies, code

In [ ]:
# --- GPU check -----------------------------------------------------------
import subprocess, torch, os, sys
print(subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True).stdout.strip() or 'no GPU')

# --- Drive ---------------------------------------------------------------
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/rocket_v28'
os.makedirs(f'{DRIVE}/models', exist_ok=True)
os.makedirs(f'{DRIVE}/logs', exist_ok=True)
print('output folder:', DRIVE)
got = sorted(os.listdir(f'{DRIVE}/models'))
print('found in models/:', got if got else '(nothing yet — will start fresh)')

# --- Dependencies --------------------------------------------------------
# tensorflow is broken on Colab Python 3.13 and SB3 does not need it; removing
# it avoids a slow import stall on every launch of the training script.
%pip install -q mujoco "stable-baselines3>=2.4" "gymnasium>=1.0" tensorboard imageio imageio-ffmpeg
%pip uninstall -y -q tensorflow tensorflow-cpu 2>/dev/null
import mujoco, gymnasium, stable_baselines3
print(f'mujoco {mujoco.__version__} | gymnasium {gymnasium.__version__} | sb3 {stable_baselines3.__version__} | cuda {torch.cuda.is_available()}')

# --- Code (always pull latest) -------------------------------------------
REPO = 'https://github.com/rajivagarwal221/Rocket-Land.git'
if not os.path.isdir('/content/rocket'):
    !git clone --depth 1 {REPO} /content/rocket
else:
    !cd /content/rocket && git fetch --depth 1 origin main -q && git reset --hard origin/main -q
%cd /content/rocket
!git log --oneline -1

## Cell 2: train — auto-resumes from Drive

Re-run this cell any time (after a disconnect, next day, different runtime). It will:
- Load `master_v28_latest.zip` and the replay buffer from Drive if they exist, and continue from that step.
- Otherwise start fresh from step 0.

Default target is **6M total steps**. Change `V28_STEPS` below if you want more or less. You can stop this cell at any time; the next run picks up at the last 100k-step checkpoint.

In [ ]:
import os
os.environ.update(
    V28_OUT   = DRIVE,
    V28_STEPS = '6000000',
    V28_NENVS = '4',
    V28_SEED  = '2828',
    V28_FRESH = '0',     # set to '1' only if you want to wipe Drive and start over
)
!python src/train_v28_mujoco.py

## Cell 3: evaluate the best checkpoint

Scores `master_v28_best.zip` over 500 seeded episodes at the ship bar (full difficulty, 0.5 m pad). The CSV is written to Drive next to the checkpoint.

In [ ]:
import os
BEST = f'{DRIVE}/models/master_v28_best.zip'
if not os.path.isfile(BEST):
    print(f'no best checkpoint yet at {BEST}. Run Cell 2 first until at least one evaluation prints "<- new best".')
else:
    !python src/evaluate_3d.py --model {BEST} --episodes 500 --device cuda \
        --out {DRIVE}/logs/v28_best_audit.csv